# Square-QDM checkerboard-family evidence for Sec. VII

This notebook implements the gated fixed-width workflow in `SEC7_NUMERICAL_PROVISIONING.md`.
It first tests the energy-density, transport, and reduced-sector gates.  The expensive
family-wide thermal scan runs only after the structural gates pass.  If the nonzero
uniform potential fails the $\beta=0$ energy-density gate, the automatic protocol uses
an energy-matched finite-$\beta$ comparator and records that the main-text $\beta=0$
claim is unavailable.

## Imports and run controls

In [ ]:
from dataclasses import replace
from itertools import product
from pathlib import Path
import sys, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as la
import scipy.sparse as sp
from scipy.optimize import brentq
from IPython.display import display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate qlinks repository root")
for extra in (REPO_ROOT, REPO_ROOT/'experimental'/'jobs'):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from helpers import (
    PRX_FOUR_PANEL_FIGSIZE, add_panel_label, orthonormalize_columns,
    projector_deleted_block_covariance, projector_resolved_energy_basis,
    save_prx_figure, set_revtex_matplotlib_style, use_integer_ticks,
    write_figure_manifest,
)
from qdm_checkerboard_large_strip import (
    canonical_typicality_scan,
    energy_matched_canonical_estimate,
    materialize_periodic_product_state_from_basis,
    packed_binary_basis_index,
    project_sparse_operator_to_sector,
    shift_invert_partial_spectrum,
    translation_permutation_from_binary_basis,
    zero_momentum_commuting_sector_basis,
)
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.caging import (
    CageClassificationConfig, CageSearchConfig, CageSearcher, LocalQDMCageSearchConfig,
    RobustQDMLocalCageSearchConfig, LocalWitnessTemplate,
    SquareQDMPeriodicProductUnitCell, SquareQDMWitnessPlacement,
    certify_local_witness_on_square_qdm_periodic_sequence,
    certify_square_qdm_periodic_product_instance,
    certify_square_qdm_periodic_product_sequence,
    classify_cage_state, commuting_cyclic_symmetry_sector_basis, diagnose_eigenpair,
    directed_transition_witness_template, evaluate_square_qdm_classification_witnesses_on_strips,
    materialize_square_qdm_periodic_product_state,
    project_operator_to_sector, project_state_to_sector,
    robust_qdm_local_cage_search, scan_square_qdm_beta_zero_energy_density,
    select_microcanonical_window_by_width, thermodynamic_energy_window_plan,
)
from qlinks.models import SquareQDMModel, qdm_plaquette_link_gauge_matrix
from qlinks.models.couplings import peierls_plaquette_coupling

TOL=1e-10
RANK_TOL=1e-9
DARK_TOL=1e-9
RUN_PROFILE='smoke'
STRICT_CLAIMS=False
TRANSPORT_REPEATS_BY_PROFILE={'smoke':(1,2,3),'known':(1,2,3),'production':(1,2,3)}
ED_REPEATS_BY_PROFILE={'smoke':(1,), 'known':(1,2), 'production':(1,2)}
DARK_CLASSIFICATION_REPEATS_BY_PROFILE={'smoke':(1,), 'known':(1,2), 'production':(1,2)}
TRANSFER_MAX_LENGTH_BY_PROFILE={'smoke':32,'known':128,'production':256}
LARGE_STRIP_EIGENPAIRS_BY_PROFILE={'smoke':64,'known':256,'production':1024}
FINITE_BETA_SAMPLES_BY_PROFILE={'smoke':2,'known':4,'production':8}
FINITE_BETA_POINTS_BY_PROFILE={'smoke':9,'known':21,'production':41}
CHECKERBOARD_TRANSPORT_REPEATS=TRANSPORT_REPEATS_BY_PROFILE[RUN_PROFILE]
CHECKERBOARD_ED_REPEATS=ED_REPEATS_BY_PROFILE[RUN_PROFILE]
DARK_CLASSIFICATION_REPEATS=DARK_CLASSIFICATION_REPEATS_BY_PROFILE[RUN_PROFILE]
CHECKERBOARD_TRANSFER_MAX_LENGTH=TRANSFER_MAX_LENGTH_BY_PROFILE[RUN_PROFILE]
CHECKERBOARD_PHASE_VALUES=(0.0,0.025,0.05,0.075,0.10)
CHECKERBOARD_POSITIVE_PHASE_VALUES=(0.025,0.05,0.075,0.10)
CHECKERBOARD_REPRESENTATIVE_PHASE=0.05
CHECKERBOARD_THERMAL_PROTOCOL='auto'  # auto, beta0, finite-beta
CHECKERBOARD_ENERGY_MATCH_TOL=1e-3
MICROCANONICAL_PREFACTORS=(0.50,0.75,1.00)
PRIMARY_WINDOW_PREFACTOR=0.75
RUN_CHECKERBOARD_THERMAL_SCAN=True
RUN_CHECKERBOARD_CONCENTRATION=True
RUN_DARK_MANIFOLD_CLASSIFICATION=True
RUN_LARGE_STRIP=False
LARGE_STRIP_REPEATS=(3,)
LARGE_STRIP_EIGENPAIRS=LARGE_STRIP_EIGENPAIRS_BY_PROFILE[RUN_PROFILE]
LARGE_STRIP_SIGMA_OFFSET=1e-6
LARGE_STRIP_EIG_TOL=1e-9
LARGE_STRIP_MAXITER=None
FINITE_BETA_TYPICALITY_SAMPLES=FINITE_BETA_SAMPLES_BY_PROFILE[RUN_PROFILE]
FINITE_BETA_BETA_MAX=0.25
FINITE_BETA_BETA_POINTS=FINITE_BETA_POINTS_BY_PROFILE[RUN_PROFILE]
FINITE_BETA_RANDOM_SEED=20260807
LARGE_STRIP_PHASE_CHECK_VALUES=()
ENERGY_BLOCK_TOL=1e-9
USE_TEX=False
SAVE_FIGURES=True
SAVE_PDF=True
FIGURE_FORMATS=('pdf','svg')
DATA_DIR=REPO_ROOT/'experimental'/'data'/'square_qdm_draft_evidence'
FIGURE_DIR=DATA_DIR/'figures'
DATA_DIR.mkdir(parents=True,exist_ok=True); FIGURE_DIR.mkdir(parents=True,exist_ok=True)
set_revtex_matplotlib_style(base_font_size=9.0,prefer_tex=USE_TEX)

def save_figure(fig,stem):
    if SAVE_FIGURES:
        save_prx_figure(fig,stem,directory=FIGURE_DIR,formats=FIGURE_FORMATS)

def project_operator_to_sector_sparse(operator, sector):
    basis=sector.basis if hasattr(sector,'basis') else sector
    return sp.csr_array(basis.conj().T @ (operator @ basis))

def translation_permutation(model, configs, *, dx=0, dy=0):
    lookup={}
    for link in model.lattice.links:
        x,y=model.lattice.sites[int(link.source)].cell
        lookup[(int(x),int(y),str(link.kind))]=int(link.id)
    transformed=np.zeros_like(configs)
    for link in model.lattice.links:
        x,y=model.lattice.sites[int(link.source)].cell
        target=lookup[((int(x)+dx)%model.lx,(int(y)+dy)%model.ly,str(link.kind))]
        transformed[:,target]=configs[:,int(link.id)]
    index={tuple(map(int,row)):i for i,row in enumerate(configs)}
    return np.asarray([index[tuple(map(int,row))] for row in transformed],dtype=np.int64)

print({'profile':RUN_PROFILE,'transport_repeats':CHECKERBOARD_TRANSPORT_REPEATS,
       'ed_repeats':CHECKERBOARD_ED_REPEATS,'large_strip':RUN_LARGE_STRIP,
       'phase_values':CHECKERBOARD_PHASE_VALUES,'thermal_protocol':CHECKERBOARD_THERMAL_PROTOCOL})


## Recover the repeated compact cage and bounded kinetic witnesses

In [ ]:
base_model=SquareQDMModel(lx=4,ly=4,boundary_condition='periodic',winding_x=0,winding_y=0,
    winding_convention='electric',coup_kin=1.0,coup_pot=1.0)
local_cfg=LocalQDMCageSearchConfig(halo_layers=0,boundary_mode='relaxed',prune_inactive_local_basis_states=True,
    tolerance=TOL,degenerate_basis_strategy='ipr',ipr_random_seed=1234)
robust_cfg=RobustQDMLocalCageSearchConfig(local_config=local_cfg,region_strategies=('stripe',),stripe_widths=(1,),
    stripe_directions=(0,1),max_regions_per_strategy=None,block_signatures=((0,2),),max_records_per_region=2,
    min_blocks=2,max_blocks=None,max_product_support_size=2048,max_paddings_per_stage=100,
    max_paddings_per_packing=10,include_sectors=True,padding_stages=('static',),tolerance=1e-9,store_full_states=False)
stripe_certified,stripe_context=robust_qdm_local_cage_search(base_model,config=robust_cfg,return_context=True)
repeatable=[]
for report_index,report in enumerate(stripe_certified.reports):
    try:
        cell=SquareQDMPeriodicProductUnitCell.from_padding(base_model,stripe_context.blocks,report.padding,repeat_axis='x')
        certificate=certify_square_qdm_periodic_product_sequence(cell)
    except ValueError:
        continue
    if certificate.is_certified:
        repeatable.append((report_index,cell,certificate))
if not repeatable: raise RuntimeError('No repeatable compact cage found')
repeatable_report_index,product_unit_cell,product_sequence=repeatable[0]
stripe_record=stripe_certified.records[repeatable_report_index]
classification=classify_cage_state(stripe_record.cage_state,kinetic_matrix=stripe_certified.kinetic_matrix,
    basis_configs=stripe_certified.basis.states,hilbert_size=stripe_certified.hilbert_size,
    config=CageClassificationConfig(sector_policy='infer_support_component'))
strip_report=evaluate_square_qdm_classification_witnesses_on_strips(classification,model=base_model,lengths=(4,8,12),
    winding_sector=(0,0),normalization='operator_norm',winding_projection='fourier')
z_reference=strip_report.records[0].witness
z_placement=strip_report.records[0].placement
op=np.asarray(z_reference.template.local_operator,dtype=np.complex128)
adj=np.abs(op)>TOL; target=int(np.argmax(np.sum(adj,axis=1))); sources=np.flatnonzero(adj[target])
a_template=directed_transition_witness_template(target_pattern=z_reference.template.local_patterns[target],
    source_patterns=[z_reference.template.local_patterns[i] for i in sources],
    amplitudes=[op[target,i] for i in sources],metadata={'name':'A_R'},normalization='operator_norm')
a_reference=a_template.instantiate(z_reference.variable_indices)
a_placement=SquareQDMWitnessPlacement.from_local_witness(base_model,a_reference)
a_cert=certify_local_witness_on_square_qdm_periodic_sequence(product_sequence,a_reference)
z_cert=certify_local_witness_on_square_qdm_periodic_sequence(product_sequence,z_reference)
pd.DataFrame([{'witness':'A','annihilation_residual':a_cert.annihilation_residual,'Q_norm':a_cert.witness.q_operator_norm},
              {'witness':'Z','annihilation_residual':z_cert.annihilation_residual,'Q_norm':z_cert.witness.q_operator_norm}]).to_csv(
              DATA_DIR/'qdm_checkerboard_AZ_certificates.csv',index=False)
print(product_sequence.to_summary_dict())

## Gate 1: fixed-width energy-density matching

In [ ]:
transfer_lengths=tuple(range(4,CHECKERBOARD_TRANSFER_MAX_LENGTH+1,4))
energy_scan=scan_square_qdm_beta_zero_energy_density([(L,4) for L in transfer_lengths],potential_coupling=1.0,
    winding_sector=(0,0),winding_projection='fourier')
cage_ed=float(product_sequence.energy_density)
energy_rows=[]
for ev in energy_scan.evaluations:
    d=ev.to_summary_dict()
    energy_rows.append({'Lx':int(ev.length),'Ly':4,'cage_energy_density':cage_ed,
        'beta0_trace_energy_density':float(ev.energy_density),
        'signed_mismatch':float(ev.energy_density-cage_ed),
        'absolute_mismatch':abs(float(ev.energy_density-cage_ed)),
        'partition_method':'exact_transfer_fourier'})
energy_match=pd.DataFrame(energy_rows)
energy_match.to_csv(DATA_DIR/'qdm_checkerboard_energy_density_match.csv',index=False)
fit_rows=[]
fit_frame=energy_match[energy_match['Lx']>=max(12,transfer_lengths[len(transfer_lengths)//4])]
L=fit_frame['Lx'].to_numpy(float); y=fit_frame['signed_mismatch'].to_numpy(float)
for name,x in [('constant',np.zeros_like(L)),('Delta_inf+c/Lx',1/L),('Delta_inf+c/Lx^2',1/L**2)]:
    if name=='constant': intercept=float(np.mean(y)); slope=0.; pred=np.full_like(y,intercept)
    else: slope,intercept=np.polyfit(x,y,1); pred=intercept+slope*x
    fit_rows.append({'fit_form':name,'included_Lx':','.join(map(str,L.astype(int))),
        'limit':float(intercept),'slope':float(slope),'rmse':float(np.sqrt(np.mean((y-pred)**2)))})
energy_fit=pd.DataFrame(fit_rows)
energy_fit['gate_tolerance']=CHECKERBOARD_ENERGY_MATCH_TOL
energy_fit.to_csv(DATA_DIR/'qdm_checkerboard_energy_density_fit.csv',index=False)
preferred=float(energy_fit.loc[energy_fit.fit_form=='Delta_inf+c/Lx','limit'].iloc[0])
BETA0_GATE_PASSED=abs(preferred)<=CHECKERBOARD_ENERGY_MATCH_TOL
ACTIVE_THERMAL_PROTOCOL=('beta0' if BETA0_GATE_PASSED else 'finite-beta') if CHECKERBOARD_THERMAL_PROTOCOL=='auto' else CHECKERBOARD_THERMAL_PROTOCOL
pd.DataFrame([{'gate':'fixed_width_energy_density','passed':BETA0_GATE_PASSED,'preferred_limit':preferred,
    'tolerance':CHECKERBOARD_ENERGY_MATCH_TOL,'selected_protocol':ACTIVE_THERMAL_PROTOCOL}]).to_csv(
    DATA_DIR/'qdm_checkerboard_gate_status.csv',index=False)
display(energy_fit); print({'beta0_gate_passed':BETA0_GATE_PASSED,'active_protocol':ACTIVE_THERMAL_PROTOCOL})

## Gate 2: checkerboard transport, local constraints, gauge quotient, and common sector

In [ ]:
def checkerboard_sign(model,pid):
    x,y=model.lattice.plaquette_anchor_cell(int(pid)); return 1 if (int(x)+int(y))%2==0 else -1

def checkerboard_instance(repeats,phase):
    raw=product_unit_cell.instantiate(int(repeats))
    model0=replace(raw.model,winding_x=0,winding_y=0)
    couplings={int(pid):peierls_plaquette_coupling(1.0,float(phase)*checkerboard_sign(model0,pid))
               for pid in model0.plaquette_ids()}
    model=replace(model0,coup_kin=couplings,coup_pot=1.0)
    return replace(raw,model=model)

family_rows=[]; constraint_rows=[]; gauge_rows=[]; sector_rows=[]
for repeats in CHECKERBOARD_TRANSPORT_REPEATS:
    Lx=4*int(repeats)
    for phase in CHECKERBOARD_PHASE_VALUES:
        instance=checkerboard_instance(repeats,phase)
        cert=certify_square_qdm_periodic_product_instance(instance,tolerance=1e-9)
        family_rows.append({'repeats':repeats,'Lx':Lx,'Ly':4,'phase':phase,'cage_residual':max([v for b in cert.block_certificates for v in (b.kinetic_residual,b.potential_residual,b.leakage_residual)],default=0.0),
            'cage_energy':float(cert.energy.real),'cage_energy_density':float(cert.energy.real/(4*Lx)),
            'winding_sector':repr(cert.winding_sector),'A_localized_residual':a_cert.annihilation_residual,
            'Z_localized_residual':z_cert.annihilation_residual,'local_certificate_passed':cert.is_certified})
    model=checkerboard_instance(repeats,0.0).model
    active=set(int(pid) for block in checkerboard_instance(repeats,0.0).blocks for pid in block.record.active_plaquette_ids)
    for x in range(Lx):
        for y in range(2):
            p1=int(model.lattice.plaquette_id_from_cell(x,y)); p2=int(model.lattice.plaquette_id_from_cell(x,(y+2)%4))
            chi1=checkerboard_sign(model,p1); chi2=checkerboard_sign(model,p2)
            constraint_rows.append({'repeats':repeats,'Lx':Lx,'row_x':x,'row_y':y,'plaquette_1':p1,'plaquette_2':p2,
                'chi_1':chi1,'chi_2':chi2,'equal_phase_constraint_passed':chi1==chi2,
                'both_active':p1 in active and p2 in active})
    chi=np.asarray([checkerboard_sign(model,pid) for pid in model.plaquette_ids()],float)
    incidence=qdm_plaquette_link_gauge_matrix(model.lattice)
    proj=incidence@np.linalg.lstsq(incidence,chi,rcond=RANK_TOL)[0]
    gauge_rows.append({'repeats':repeats,'Lx':Lx,'plaquette_count':len(chi),
        'link_gauge_rank':int(np.linalg.matrix_rank(incidence,tol=RANK_TOL)),
        'checkerboard_norm':float(np.linalg.norm(chi)),'distance_from_link_gauge_image':float(np.linalg.norm(chi-proj)),
        'relative_distance':float(np.linalg.norm(chi-proj)/np.linalg.norm(chi))})
    if repeats in CHECKERBOARD_ED_REPEATS:
        instance=checkerboard_instance(repeats,CHECKERBOARD_REPRESENTATIVE_PHASE)
        build=instance.model.build(basis_solver='dfs',builder='bitmask',backend='scipy',sort_basis=True)
        configs=basis_configs_from_build_result(build); cage=materialize_square_qdm_periodic_product_state(instance,configs)
        tx2=translation_permutation(instance.model,configs,dx=2); ty2=translation_permutation(instance.model,configs,dy=2)
        sector=commuting_cyclic_symmetry_sector_basis((tx2,ty2),orders=(Lx//2,2),momentum_indices=(0,0),
            labels={'Tx2_k':0,'Ty2_k':0})
        projected=project_state_to_sector(cage,sector); pnorm=float(np.linalg.norm(projected))
        if pnorm>TOL: projected/=pnorm
        qvals={}
        for name,placement in [('A',a_placement),('Z',z_placement)]:
            op=placement.instantiate_on_model(instance.model).embed(configs)
            q=project_operator_to_sector(op.conj().T@op,sector)
            qvals[name]=float(np.vdot(projected,q@projected).real)
        sector_rows.append({'repeats':repeats,'Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
            'Tx2_order':Lx//2,'Ty2_order':2,'kx2_index':0,'ky2_index':0,
            'sector_dimension':sector.sector_dimension,'cage_projection_norm':pnorm,
            'projected_QA':qvals['A'],'projected_QZ':qvals['Z'],'status':'verified_ED'})
    else:
        sector_rows.append({'repeats':repeats,'Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
            'Tx2_order':Lx//2,'Ty2_order':2,'kx2_index':0,'ky2_index':0,
            'sector_dimension':np.nan,'cage_projection_norm':np.nan,'projected_QA':np.nan,'projected_QZ':np.nan,
            'status':'local_transport_verified_sector_projection_pending_large_strip'})
family=pd.DataFrame(family_rows); constraints=pd.DataFrame(constraint_rows); gauge=pd.DataFrame(gauge_rows); common_sector=pd.DataFrame(sector_rows)
family.to_csv(DATA_DIR/'qdm_checkerboard_fixed_width_family.csv',index=False)
constraints.to_csv(DATA_DIR/'qdm_checkerboard_compatibility_constraints.csv',index=False)
gauge.to_csv(DATA_DIR/'qdm_checkerboard_gauge_quotient.csv',index=False)
common_sector.to_csv(DATA_DIR/'qdm_checkerboard_common_symmetry_sector.csv',index=False)
GATE2_LOCAL_PASSED=bool(family.local_certificate_passed.all() and constraints.equal_phase_constraint_passed.all())
display(family); display(common_sector)

### Reference Type-I continuation inventory

In [ ]:
reference_build=base_model.build(basis_solver='dfs',builder='sparse',backend='scipy',sort_basis=True)
reference_search=CageSearcher.from_model_build_result(reference_build,config=CageSearchConfig(
    search_type='type1',tolerance=TOL,degenerate_basis_strategy='ipr',ipr_n_restarts=32,
    ipr_candidate_count=24,ipr_random_seed=1234)).run()
type1_rows=[]
for signature in reference_search.signatures:
    for record_index,record in enumerate(reference_search[signature]):
        state=np.zeros(reference_search.hilbert_size,dtype=np.complex128)
        state[np.asarray(record.cage_state.support,dtype=np.int64)]=record.cage_state.local_state
        for phase in CHECKERBOARD_PHASE_VALUES:
            model=checkerboard_instance(1,phase).model
            build=model.build(basis_solver='dfs',builder='sparse',backend='scipy',sort_basis=True)
            np.testing.assert_array_equal(build.basis.states,reference_build.basis.states)
            report=diagnose_eigenpair(build.hamiltonian,state)
            type1_rows.append({'signature':repr(signature),'record_index':record_index,'support_size':len(record.cage_state.support),
                'phase':phase,'energy_real':float(report.energy.real),'energy_imag':float(report.energy.imag),
                'residual':float(report.residual_norm),'status':'exact' if report.residual_norm<=1e-8 else 'lifted',
                'inventory_scope':'4x4 reference Type-I states continued along checkerboard path'})
type1=pd.DataFrame(type1_rows)
type1.to_csv(DATA_DIR/'qdm_checkerboard_type1_continuation.csv',index=False)
display(type1.groupby(['phase','status']).size().rename('count').reset_index())

## Complete stripe-local algebra and translated joint-dark machinery

In [ ]:
def stripe_algebra(configs,model,sector):
    witness=z_placement.instantiate_on_model(model); variables=tuple(map(int,witness.variable_indices))
    patterns=np.unique(configs[:,variables],axis=0)
    position={v:i for i,v in enumerate(variables)}
    site_lookup={tuple(site.cell):int(site.id) for site in model.lattice.sites}
    boundary=[]
    for cell in z_placement.affected_sites:
        sid=site_lookup[tuple(cell)]
        inc=[int(model.layout.link_variable_index(link)) for link in model.lattice.incident_links(sid)]
        if not all(v in position for v in inc): boundary.append(tuple(inc))
    signatures=[]
    for pattern in patterns:
        signatures.append(tuple(sum(int(pattern[position[v]]) for v in inc if v in position) for inc in boundary))
    groups={}
    for i,sig in enumerate(signatures): groups.setdefault(sig,[]).append(i)
    matrices=[]; names=[]
    n=len(patterns)
    for block_id,indices in enumerate(groups.values()):
        for i in indices:
            m=np.zeros((n,n),complex); m[i,i]=1; matrices.append(m); names.append(f'b{block_id}_d{i}')
        for pos,i in enumerate(indices):
            for j in indices[pos+1:]:
                m=np.zeros((n,n),complex); m[i,j]=m[j,i]=1/np.sqrt(2); matrices.append(m); names.append(f'b{block_id}_s{i}_{j}')
                m=np.zeros((n,n),complex); m[i,j]=-1j/np.sqrt(2); m[j,i]=1j/np.sqrt(2); matrices.append(m); names.append(f'b{block_id}_a{i}_{j}')
    projected=[]; kept=[]
    for name,matrix in zip(names,matrices,strict=True):
        template=LocalWitnessTemplate(pattern_key=(),local_patterns=tuple(tuple(map(int,p)) for p in patterns),
            local_operator=matrix,metadata={'name':name,'boundary_flux_blocks':len(groups)})
        full=template.instantiate(variables).embed(configs)
        op=project_operator_to_sector_sparse(full,sector)
        norm=float(np.sqrt(np.real(op.conj().multiply(op).sum())))
        if norm>1e-12: projected.append(op/norm); kept.append(name)
    return projected,kept,{'local_pattern_count':len(patterns),'boundary_flux_block_count':len(groups),
        'boundary_flux_block_dimensions':repr(sorted(map(len,groups.values()))),'formal_operator_dimension':sum(len(v)**2 for v in groups.values()),
        'projected_operator_dimension':len(projected)}

def translated_joint_dark(configs,model,sector):
    total=None
    for x in range(model.lx):
        for placement in (a_placement,z_placement):
            op=placement.instantiate_on_model(model,origin_x=x).embed(configs)
            q=op.conj().T@op
            total=q if total is None else total+q
    return project_operator_to_sector_sparse(total,sector)

def joint_dark_kernel(energies,vectors,q_all,tower):
    order=np.argsort(np.asarray(energies,float)); energies=np.asarray(energies,float)[order]; vectors=np.asarray(vectors)[:,order]
    groups=[]
    for i,e in enumerate(energies):
        if not groups or abs(e-energies[groups[-1][-1]])>ENERGY_BLOCK_TOL: groups.append([i])
        else: groups[-1].append(i)
    cols=[]; rows=[]
    for bid,g in enumerate(groups):
        basis=vectors[:,g]; comp=basis.conj().T@(q_all@basis); comp=0.5*(comp+comp.conj().T)
        vals,rot=la.eigh(comp,check_finite=False); scale=max(1.,float(np.max(np.abs(vals),initial=0.)))
        keep=np.flatnonzero(vals<=DARK_TOL*scale); dark=basis@rot[:,keep] if keep.size else np.zeros((basis.shape[0],0),complex)
        weight=float(np.linalg.norm(dark.conj().T@tower)**2) if keep.size else 0.
        cols.extend(dark[:,i] for i in range(dark.shape[1]))
        rows.append({'energy_block_id':bid,'energy':float(np.mean(energies[g])),'block_dimension':len(g),
            'joint_dark_rank':int(keep.size),'target_weight':weight,'remaining_rank_after_target':max(0,int(keep.size)-(weight>1-1e-7))})
    exceptional=orthonormalize_columns(np.column_stack(cols),tolerance=1e-9) if cols else np.zeros((vectors.shape[0],0),complex)
    return exceptional,rows

def canonical_weights(energies,target):
    energies=np.asarray(energies,float)
    def weights(beta):
        x=-beta*energies; x-=np.max(x); w=np.exp(x); return w/w.sum()
    f0=float(np.mean(energies)-target)
    if abs(f0)<1e-12:return 0.,weights(0.)
    direction=1. if f0>0 else -1.; bound=direction
    while abs(bound)<128 and (np.dot(weights(bound),energies)-target)*f0>0: bound*=2
    if abs(bound)>=128: raise RuntimeError('Could not bracket finite-beta match')
    beta=float(brentq(lambda b:np.dot(weights(b),energies)-target,*sorted((0.,bound))))
    return beta,weights(beta)

def subspace_comparison(candidate,reference,tolerance=1e-8):
    candidate=orthonormalize_columns(candidate,tolerance=tolerance) if candidate.size else np.zeros((reference.shape[0],0),complex)
    reference=orthonormalize_columns(reference,tolerance=tolerance) if reference.size else np.zeros((candidate.shape[0],0),complex)
    if candidate.shape[0] != reference.shape[0]: raise ValueError('subspaces have incompatible ambient dimensions')
    if candidate.shape[1]:
        unexplained=reference-candidate@(candidate.conj().T@reference)
    else: unexplained=reference.copy()
    if reference.shape[1]:
        non_dark=candidate-reference@(reference.conj().T@candidate)
    else: non_dark=candidate.copy()
    return {'candidate_rank':candidate.shape[1],'reference_rank':reference.shape[1],
        'unexplained_reference_norm':float(np.linalg.norm(unexplained)),
        'candidate_outside_reference_norm':float(np.linalg.norm(non_dark)),
        'explained':bool(np.linalg.norm(unexplained)<=tolerance*max(1.,np.sqrt(reference.shape[1])))}


## Gate 3 and energy-resolved family pilot

In [ ]:
thermal_rows=[]; beta0_rows=[]; scatter_frames=[]; dark_rows=[]; concentration_rows=[]; trace_rows=[]; cleaning_rows=[]
finite_beta_exact_rows=[]; representative_contexts={}
if RUN_CHECKERBOARD_THERMAL_SCAN and GATE2_LOCAL_PASSED:
  for repeats in CHECKERBOARD_ED_REPEATS:
    for phase in CHECKERBOARD_PHASE_VALUES:
      instance=checkerboard_instance(repeats,phase); model=instance.model; Lx=model.lx; volume=Lx*4
      build=model.build(basis_solver='dfs',builder='bitmask',backend='scipy',sort_basis=True); configs=basis_configs_from_build_result(build)
      cage=materialize_square_qdm_periodic_product_state(instance,configs)
      tx2=translation_permutation(model,configs,dx=2); ty2=translation_permutation(model,configs,dy=2)
      sector=commuting_cyclic_symmetry_sector_basis((tx2,ty2),orders=(Lx//2,2),momentum_indices=(0,0))
      tower=project_state_to_sector(cage,sector); tower/=np.linalg.norm(tower)
      h=project_operator_to_sector(build.hamiltonian,sector); energies,vectors=la.eigh(h,check_finite=False)
      local={name:placement.instantiate_on_model(model).embed(configs) for name,placement in [('A',a_placement),('Z',z_placement)]}
      q={name:project_operator_to_sector(op.conj().T@op,sector) for name,op in local.items()}
      raw_Q={name:np.real(np.einsum('ij,ij->j',vectors.conj(),op@vectors)) for name,op in q.items()}
      q_all=translated_joint_dark(configs,model,sector); exceptional,jrows=joint_dark_kernel(energies,vectors,q_all,tower)
      for r in jrows:r.update({'repeats':repeats,'Lx':Lx,'phase':phase,'spectrum_method':'full_ED'})
      dark_rows.extend(jrows)
      resolved=projector_resolved_energy_basis(energies,vectors,exceptional,energy_tolerance=ENERGY_BLOCK_TOL,vector_tolerance=1e-9)
      keep=~resolved['is_exceptional'].astype(bool); clean_E=resolved['energies'][keep]
      clean_Q={name:np.real(np.einsum('ij,ij->j',resolved['basis'][:,keep].conj(),op@resolved['basis'][:,keep])) for name,op in q.items()}
      target=float(np.vdot(tower,h@tower).real)
      beta0_weights=np.full(clean_E.size,1/clean_E.size); raw_beta0_weights=np.full(energies.size,1/energies.size)
      beta0_reference={name:float(np.dot(beta0_weights,vals)) for name,vals in clean_Q.items()}
      beta0_reference_raw={name:float(np.dot(raw_beta0_weights,vals)) for name,vals in raw_Q.items()}
      beta_clean,finite_beta_weights=canonical_weights(clean_E,target)
      beta_raw,finite_beta_weights_raw=canonical_weights(energies,target)
      finite_beta_clean={name:float(np.dot(finite_beta_weights,vals)) for name,vals in clean_Q.items()}
      finite_beta_raw={name:float(np.dot(finite_beta_weights_raw,vals)) for name,vals in raw_Q.items()}
      finite_beta_exact_rows.append({'repeats':repeats,'Lx':Lx,'Ly':4,'phase':phase,'method':'exact_common_sector_ED',
        'sector_dimension':sector.sector_dimension,'target_energy':target,'target_energy_density':target/volume,
        'beta_clean':beta_clean,'beta_raw':beta_raw,'tau_A_clean':finite_beta_clean['A'],'tau_Z_clean':finite_beta_clean['Z'],
        'tau_A_raw':finite_beta_raw['A'],'tau_Z_raw':finite_beta_raw['Z'],'stochastic_samples':0,
        'beta_stderr':0.0,'tau_A_stderr':0.0,'tau_Z_stderr':0.0})
      if ACTIVE_THERMAL_PROTOCOL=='beta0':
          beta=0.; beta_for_raw=0.
          reference_clean=beta0_reference; reference_raw=beta0_reference_raw
      else:
          beta=beta_clean; beta_for_raw=beta_raw
          reference_clean=finite_beta_clean; reference_raw=finite_beta_raw
      # Preserve the established clean--clean finite-size estimator in Delta.
      # The physical raw canonical target is exported in parallel for the
      # fixed-width finite-temperature limit.
      reference=reference_clean
      trace_rows.append({'repeats':repeats,'Lx':Lx,'phase':phase,'sector_dimension':sector.sector_dimension,
        'beta0_energy_density_clean':float(np.dot(beta0_weights,clean_E)/volume),'target_energy_density':target/volume,
        'tau_A_beta0_clean':float(np.dot(beta0_weights,clean_Q['A'])),'tau_Z_beta0_clean':float(np.dot(beta0_weights,clean_Q['Z'])),
        'tau_A_transfer':np.nan,'tau_Z_transfer':np.nan})
      primary_indices=None
      for pref in MICROCANONICAL_PREFACTORS:
        plan=thermodynamic_energy_window_plan(volume=volume,energy_density=target/volume,width_prefactor=pref,local_energy_scale=1.)
        window=select_microcanonical_window_by_width(clean_E,target_energy=target,half_width=plan.half_width,degeneracy_tolerance=ENERGY_BLOCK_TOL)
        idx=np.asarray(window.indices,int); mc={name:float(np.mean(vals[idx])) for name,vals in clean_Q.items()}
        raw_window=select_microcanonical_window_by_width(energies,target_energy=target,half_width=plan.half_width,degeneracy_tolerance=ENERGY_BLOCK_TOL)
        raw_idx=np.asarray(raw_window.indices,int); mc_raw={name:float(np.mean(vals[raw_idx])) for name,vals in raw_Q.items()}
        row={'repeats':repeats,'Lx':Lx,'Ly':4,'phase':phase,'thermal_protocol':ACTIVE_THERMAL_PROTOCOL,'matched_beta':beta,
          'sector_dimension':sector.sector_dimension,'cage_energy':target,'cage_energy_density':target/volume,
          'cage_residual':float(np.linalg.norm(h@tower-target*tower)),'window_prefactor':pref,'window_half_width':window.half_width,
          'window_energy_density_half_width':window.half_width/volume,'window_state_count':window.n_states,'window_coverage_complete':True,
          'spectrum_method':'full_ED','reference_method':'exact_common_sector_ED','reference_cleaning':'clean',
          'joint_dark_rank':exceptional.shape[1],'removed_fraction':float(exceptional.shape[1]/max(1,raw_window.n_states)),
          'raw_window_state_count':raw_window.n_states,'clean_window_state_count':window.n_states,
          'tau_A_mc':mc['A'],'tau_Z_mc':mc['Z'],'tau_A_mc_raw':mc_raw['A'],'tau_Z_mc_raw':mc_raw['Z'],
          'tau_A_reference':reference['A'],'tau_Z_reference':reference['Z'],
          'tau_A_reference_raw':reference_raw['A'],'tau_Z_reference_raw':reference_raw['Z'],
          'tau_A_reference_clean':reference_clean['A'],'tau_Z_reference_clean':reference_clean['Z'],
          'tau_A_reference_physical':reference_raw['A'],'tau_Z_reference_physical':reference_raw['Z'],'matched_beta_raw':beta_for_raw,
          'delta_A':abs(mc['A']-reference['A']),'delta_Z':abs(mc['Z']-reference['Z']),
          'delta_A_clean_clean':abs(mc['A']-reference_clean['A']),'delta_Z_clean_clean':abs(mc['Z']-reference_clean['Z']),
          'delta_A_physical_target':abs(mc['A']-reference_raw['A']),'delta_Z_physical_target':abs(mc['Z']-reference_raw['Z']),
          'Delta_physical_target':max(abs(mc['A']-reference_raw['A']),abs(mc['Z']-reference_raw['Z'])),
          'Delta':max(abs(mc['A']-reference['A']),abs(mc['Z']-reference['Z'])),
          'cage_QA':float(np.vdot(tower,q['A']@tower).real),'cage_QZ':float(np.vdot(tower,q['Z']@tower).real)}
        thermal_rows.append(row)
        beta0_row=dict(row)
        beta0_row.update({'thermal_protocol':'beta0_gate_failed_diagnostic' if not BETA0_GATE_PASSED else 'beta0_diagnostic',
            'matched_beta':0.0,'matched_beta_raw':0.0,'reference_method':'exact_beta0_common_sector',
            'tau_A_reference':beta0_reference['A'],'tau_Z_reference':beta0_reference['Z'],
            'tau_A_reference_raw':beta0_reference_raw['A'],'tau_Z_reference_raw':beta0_reference_raw['Z'],
            'tau_A_reference_clean':beta0_reference['A'],'tau_Z_reference_clean':beta0_reference['Z'],
            'tau_A_reference_physical':beta0_reference_raw['A'],'tau_Z_reference_physical':beta0_reference_raw['Z'],
            'delta_A':abs(mc['A']-beta0_reference['A']),'delta_Z':abs(mc['Z']-beta0_reference['Z']),
            'delta_A_clean_clean':abs(mc['A']-beta0_reference['A']),'delta_Z_clean_clean':abs(mc['Z']-beta0_reference['Z']),
            'delta_A_physical_target':abs(mc['A']-beta0_reference_raw['A']),'delta_Z_physical_target':abs(mc['Z']-beta0_reference_raw['Z']),
            'Delta_physical_target':max(abs(mc['A']-beta0_reference_raw['A']),abs(mc['Z']-beta0_reference_raw['Z'])),
            'Delta':max(abs(mc['A']-beta0_reference['A']),abs(mc['Z']-beta0_reference['Z']))})
        beta0_rows.append(beta0_row)
        cleaning_rows.append({'repeats':repeats,'Lx':Lx,'phase':phase,'window_prefactor':pref,
            'raw_window_state_count':raw_window.n_states,'clean_window_state_count':window.n_states,
            'removed_joint_dark_rank':exceptional.shape[1],'removed_fraction':float(exceptional.shape[1]/max(1,raw_window.n_states)),
            'tau_A_mc_raw':mc_raw['A'],'tau_A_mc_clean':mc['A'],'tau_Z_mc_raw':mc_raw['Z'],'tau_Z_mc_clean':mc['Z'],
            'tau_A_reference_raw':reference_raw['A'],'tau_A_reference_clean':reference_clean['A'],
            'tau_Z_reference_raw':reference_raw['Z'],'tau_Z_reference_clean':reference_clean['Z'],
            'energy_block_tolerance':ENERGY_BLOCK_TOL,'dark_tolerance':DARK_TOL})
        if abs(pref-PRIMARY_WINDOW_PREFACTOR)<TOL: primary_indices=idx
      if primary_indices is not None:
        scatter=pd.DataFrame({'repeats':repeats,'Lx':Lx,'phase':phase,'energy':clean_E,
            'energy_density':clean_E/volume,'Q_A':clean_Q['A'],'Q_Z':clean_Q['Z'],'is_tower_state':False})
        scatter_frames.append(scatter)
        if RUN_CHECKERBOARD_CONCENTRATION:
          ops,names,meta=stripe_algebra(configs,model,sector)
          covariance=projector_deleted_block_covariance(energies,vectors,exceptional,ops,
              np.asarray(select_microcanonical_window_by_width(energies,target_energy=target,
              half_width=thermodynamic_energy_window_plan(volume=volume,energy_density=target/volume,
              width_prefactor=PRIMARY_WINDOW_PREFACTOR,local_energy_scale=1.).half_width,
              degeneracy_tolerance=ENERGY_BLOCK_TOL).indices,int),energy_tolerance=ENERGY_BLOCK_TOL,vector_tolerance=1e-9)
          concentration_rows.append({'repeats':repeats,'Lx':Lx,'phase':phase,'operator_space_dimension':len(ops),
              **meta,'largest_covariance_eigenvalue':covariance['largest_eigenvalue'],'w':covariance['largest_width'],
              'median_nonidentity_width':covariance['median_nonidentity_width'],'energy_block_tolerance':ENERGY_BLOCK_TOL,
              'spectrum_method':'full_ED','window_coverage_complete':True,
              'worst_coefficients':repr(dict(zip(names,np.asarray(covariance['worst_coefficients']).tolist(),strict=True)))})
      if np.isclose(phase,CHECKERBOARD_REPRESENTATIVE_PHASE) and repeats in DARK_CLASSIFICATION_REPEATS:
          representative_contexts[repeats]={'instance':instance,'model':model,'build':build,'configs':configs,'sector':sector,
              'tower':tower,'energies':energies,'vectors':vectors,'exceptional':exceptional,'q_all':q_all}
thermal=pd.DataFrame(thermal_rows); beta0=pd.DataFrame(beta0_rows); scatter=pd.concat(scatter_frames,ignore_index=True) if scatter_frames else pd.DataFrame()
dark=pd.DataFrame(dark_rows); concentration=pd.DataFrame(concentration_rows); traces=pd.DataFrame(trace_rows); cleaning=pd.DataFrame(cleaning_rows)
finite_beta_exact=pd.DataFrame(finite_beta_exact_rows)
beta0.to_csv(DATA_DIR/'qdm_checkerboard_beta0_overlap.csv',index=False)
thermal.to_csv(DATA_DIR/'qdm_checkerboard_window_systematics.csv',index=False)
dark.to_csv(DATA_DIR/'qdm_checkerboard_joint_dark_kernel.csv',index=False)
cleaning.to_csv(DATA_DIR/'qdm_checkerboard_cleaning_audit.csv',index=False)
thermal.to_csv(DATA_DIR/'qdm_checkerboard_thermal_overlap.csv',index=False)
concentration.to_csv(DATA_DIR/'qdm_checkerboard_concentration_grid.csv',index=False)
concentration.to_csv(DATA_DIR/'qdm_checkerboard_worst_eigenoperator.csv',index=False)
traces.to_csv(DATA_DIR/'qdm_checkerboard_resolved_beta0_trace.csv',index=False)
if not scatter.empty:
    rep=scatter[np.isclose(scatter.phase,CHECKERBOARD_REPRESENTATIVE_PHASE)]
    Lmax=int(rep.Lx.max()); rep[rep.Lx==Lmax].to_csv(DATA_DIR/'qdm_checkerboard_eth_scatter.csv',index=False)
from qlinks.caging import SquareQDMStripTransferMatrix
transfer=SquareQDMStripTransferMatrix(circumference=4); overlap_rows=[]
for name,placement in [('A',a_placement),('Z',z_placement)]:
    scaling=transfer.scan_witness(placement,lengths=tuple(4*r for r in CHECKERBOARD_ED_REPEATS),boundary_x='periodic',
        winding_sector=(0,0),winding_projection='fourier')
    for ev in scaling.evaluations: overlap_rows.append({'witness':name,'Lx':ev.length,'transfer_target':ev.expectation})
transfer_targets=pd.DataFrame(overlap_rows)
if not traces.empty:
    for i,row in traces.iterrows():
        for name in ('A','Z'):
            match=transfer_targets[(transfer_targets.witness==name)&(transfer_targets.Lx==row.Lx)]
            if len(match): traces.loc[i,f'tau_{name}_transfer']=float(match.transfer_target.iloc[0])
for name in ('A','Z'):
    traces[f'delta_{name}_resolved_transfer']=np.abs(traces[f'tau_{name}_beta0_clean']-traces[f'tau_{name}_transfer']) if not traces.empty else np.nan
traces.to_csv(DATA_DIR/'qdm_checkerboard_resolved_beta0_trace.csv',index=False)
traces.to_csv(DATA_DIR/'qdm_checkerboard_transfer_sector_overlap.csv',index=False)
if not finite_beta_exact.empty:
    phase_check=finite_beta_exact.copy()
    for _, group in phase_check.groupby('Lx'):
        rep_index=group.index[int(np.argmin(np.abs(group.phase.to_numpy()-CHECKERBOARD_REPRESENTATIVE_PHASE)))]
        for key in ('beta_raw','tau_A_raw','tau_Z_raw'):
            phase_check.loc[group.index,f'{key}_minus_representative']=group[key]-phase_check.loc[rep_index,key]
    phase_check.to_csv(DATA_DIR/'qdm_checkerboard_finite_beta_transfer_phase_check.csv',index=False)
display(thermal[thermal.window_prefactor==PRIMARY_WINDOW_PREFACTOR] if not thermal.empty else thermal)


## P0.1--P0.2: finite-$\beta$ target and compact dark-manifold classification

The physical finite-temperature target is the **raw canonical trace** in the common reduced-symmetry sector. Joint-dark cleaning is retained as a finite-size diagnostic and is never silently substituted for the thermal target. The surviving translated $A,Z$ dark subspace is compared basis-independently with the projected Type-I compact-cage span.


In [ ]:
finite_beta_target_rows=[]
if not finite_beta_exact.empty:
    rep_exact=finite_beta_exact[np.isclose(finite_beta_exact.phase,CHECKERBOARD_REPRESENTATIVE_PHASE)].sort_values('Lx')
    for _,row in rep_exact.iterrows():
        finite_beta_target_rows.append({'record_type':'finite_size','Lx':int(row.Lx),'Ly':4,'phase':float(row.phase),
            'method':row.method,'target_energy_density':float(row.target_energy_density),'beta_star':float(row.beta_raw),
            'beta_stderr':0.0,'tau_A_target':float(row.tau_A_raw),'tau_Z_target':float(row.tau_Z_raw),
            'tau_A_stderr':0.0,'tau_Z_stderr':0.0,'sector_dimension':int(row.sector_dimension),'status':'exact_finite_size'})
finite_beta_target=pd.DataFrame(finite_beta_target_rows)
finite_beta_target.to_csv(DATA_DIR/'qdm_checkerboard_finite_beta_transfer_target.csv',index=False)
if not finite_beta_target.empty:
    finite_beta_target[['Lx','target_energy_density','beta_star','method','status']].to_csv(
        DATA_DIR/'qdm_checkerboard_finite_beta_energy_match.csv',index=False)
else:
    pd.DataFrame(columns=['Lx','target_energy_density','beta_star','method','status']).to_csv(
        DATA_DIR/'qdm_checkerboard_finite_beta_energy_match.csv',index=False)

compact_rows=[]; dark_compare_rows=[]
if RUN_DARK_MANIFOLD_CLASSIFICATION:
  for repeats in DARK_CLASSIFICATION_REPEATS:
    context=representative_contexts.get(int(repeats))
    if context is None:
        dark_compare_rows.append({'repeats':repeats,'Lx':4*int(repeats),'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
            'method':'direct_type1_inventory_in_common_sector','status':'context_unavailable','type1_projected_rank':np.nan,
            'joint_dark_rank':np.nan,'unexplained_joint_dark_norm':np.nan,'candidate_outside_joint_dark_norm':np.nan,'all_joint_dark_explained':False})
        continue
    search=CageSearcher.from_model_build_result(context['build'],config=CageSearchConfig(
        search_type='type1',tolerance=TOL,degenerate_basis_strategy='ipr',ipr_n_restarts=32,
        ipr_candidate_count=24,ipr_random_seed=1234)).run()
    projected_columns=[]
    for signature in search.signatures:
      for record_index,record in enumerate(search[signature]):
        state=np.zeros(search.hilbert_size,dtype=np.complex128)
        state[np.asarray(record.cage_state.support,dtype=np.int64)]=record.cage_state.local_state
        report=diagnose_eigenpair(context['build'].hamiltonian,state)
        coordinates=project_state_to_sector(state,context['sector']); projection_norm=float(np.linalg.norm(coordinates))
        if projection_norm>TOL: coordinates/=projection_norm
        q_dark=float(np.vdot(coordinates,context['q_all']@coordinates).real) if projection_norm>TOL else np.nan
        exact=bool(report.residual_norm<=1e-8 and projection_norm>TOL and q_dark<=1e-8)
        if exact: projected_columns.append(coordinates)
        compact_rows.append({'repeats':repeats,'Lx':4*int(repeats),'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
            'signature':repr(signature),'record_index':record_index,'support_size':len(record.cage_state.support),
            'energy':float(report.energy.real),'eigen_residual':float(report.residual_norm),'projection_norm':projection_norm,
            'projected_Qall':q_dark,'included_in_compact_dark_span':exact,'inventory_method':'direct_type1_search'})
    candidate=orthonormalize_columns(np.column_stack(projected_columns),tolerance=1e-9) if projected_columns else np.zeros_like(context['exceptional'][:,:0])
    comparison=subspace_comparison(candidate,context['exceptional'],tolerance=1e-7)
    dark_compare_rows.append({'repeats':repeats,'Lx':4*int(repeats),'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
        'method':'direct_type1_inventory_in_common_sector','status':'classified' if comparison['explained'] else 'unexplained_dark_remains',
        'type1_projected_rank':comparison['candidate_rank'],'joint_dark_rank':comparison['reference_rank'],
        'unexplained_joint_dark_norm':comparison['unexplained_reference_norm'],
        'candidate_outside_joint_dark_norm':comparison['candidate_outside_reference_norm'],
        'all_joint_dark_explained':comparison['explained']})
compact_dark=pd.DataFrame(compact_rows); dark_vs_type1=pd.DataFrame(dark_compare_rows)
compact_dark.to_csv(DATA_DIR/'qdm_checkerboard_compact_dark_manifold.csv',index=False)
dark_vs_type1.to_csv(DATA_DIR/'qdm_checkerboard_joint_dark_vs_type1.csv',index=False)
display(dark_vs_type1)


## P0.3--P0.4: sparse $12\times4$ common sector and third energy-resolved strip

The $12\times4$ lane is intentionally experimental. It uses a sparse $T_x^2,T_y^2$ zero-momentum projection, canonical typicality for the finite-$\beta$ trace, and shift-invert eigenpairs near the cage energy. Every row carries method, residual, stochastic-error, and window-coverage metadata. A partial spectrum is not used when it fails to cover the declared microcanonical window.


In [ ]:
large_strip_rows=[]; large_phase_rows=[]; large_dark_rows=[]; large_concentration_rows=[]; large_scatter_frames=[]
if RUN_LARGE_STRIP:
  for repeats in LARGE_STRIP_REPEATS:
    instance=checkerboard_instance(int(repeats),CHECKERBOARD_REPRESENTATIVE_PHASE); model=instance.model
    Lx=int(model.lx); volume=4*Lx; target=float(Lx)
    print({'large_strip_stage':'build','Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE})
    build=model.build(basis_solver='dfs',builder='bitmask',backend='scipy',sort_basis=True); basis=build.basis; configs=basis.states
    packed_index=packed_binary_basis_index(basis)
    tx2=translation_permutation_from_binary_basis(model,basis,packed_index=packed_index,dx=2,chunk_size=16384)
    ty2=translation_permutation_from_binary_basis(model,basis,packed_index=packed_index,dy=2,chunk_size=16384)
    sector=zero_momentum_commuting_sector_basis((tx2,ty2),labels={'Tx2_k':0,'Ty2_k':0,'Tx2_order':Lx//2,'Ty2_order':2})
    cage_full=materialize_periodic_product_state_from_basis(instance,basis)
    tower=project_state_to_sector(cage_full,sector); projection_norm=float(np.linalg.norm(tower)); tower/=projection_norm
    projected_full=np.asarray(sector.basis@tower).reshape(-1)
    localized={name:placement.instantiate_on_model(model).embed(configs) for name,placement in [('A',a_placement),('Z',z_placement)]}
    projected_q={name:project_sparse_operator_to_sector(op.conj().T@op,sector) for name,op in localized.items()}
    q_res={name:float(np.vdot(projected_full,(op.conj().T@(op@projected_full))).real) for name,op in localized.items()}
    common_sector=pd.read_csv(DATA_DIR/'qdm_checkerboard_common_symmetry_sector.csv')
    mask=(common_sector.Lx==Lx)&np.isclose(common_sector.phase,CHECKERBOARD_REPRESENTATIVE_PHASE)
    update={'repeats':repeats,'Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,'Tx2_order':Lx//2,'Ty2_order':2,
        'kx2_index':0,'ky2_index':0,'sector_dimension':sector.sector_dimension,'cage_projection_norm':projection_norm,
        'projected_QA':q_res['A'],'projected_QZ':q_res['Z'],'status':'verified_sparse_large_strip'}
    if mask.any():
        for key,value in update.items(): common_sector.loc[mask,key]=value
    else: common_sector=pd.concat([common_sector,pd.DataFrame([update])],ignore_index=True)
    common_sector.to_csv(DATA_DIR/'qdm_checkerboard_common_symmetry_sector.csv',index=False)
    h_sector=project_sparse_operator_to_sector(build.hamiltonian,sector)
    tower_energy=float(np.vdot(tower,h_sector@tower).real); tower_residual=float(np.linalg.norm(h_sector@tower-tower_energy*tower))
    print({'large_strip_stage':'canonical_typicality','sector_dimension':sector.sector_dimension,'tower_residual':tower_residual})
    scan=canonical_typicality_scan(h_sector,projected_q,beta_max=FINITE_BETA_BETA_MAX,beta_points=FINITE_BETA_BETA_POINTS,
        n_samples=FINITE_BETA_TYPICALITY_SAMPLES,random_seed=FINITE_BETA_RANDOM_SEED+Lx)
    matched=energy_matched_canonical_estimate(scan,target_energy=tower_energy)
    target_row={'record_type':'finite_size','Lx':Lx,'Ly':4,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
        'method':'canonical_typicality_common_sector','target_energy_density':tower_energy/volume,'beta_star':matched.beta,
        'beta_stderr':matched.energy_stderr/max(1e-12,abs(np.gradient(scan.energy,scan.beta)[np.argmin(abs(scan.beta-matched.beta))])),
        'tau_A_target':matched.observables['A'],'tau_Z_target':matched.observables['Z'],
        'tau_A_stderr':matched.observable_stderr['A'],'tau_Z_stderr':matched.observable_stderr['Z'],
        'sector_dimension':sector.sector_dimension,'stochastic_samples':scan.n_samples,'random_seed':scan.random_seed,
        'beta_bracket':repr(matched.bracket),'status':'typicality_finite_size'}
    finite_beta_target=pd.concat([finite_beta_target,pd.DataFrame([target_row])],ignore_index=True)
    large_phase_rows.append({'Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,'method':'canonical_typicality_common_sector',
        'beta_raw':matched.beta,'tau_A_raw':matched.observables['A'],'tau_Z_raw':matched.observables['Z'],
        'beta_stderr':target_row['beta_stderr'],'tau_A_stderr':matched.observable_stderr['A'],'tau_Z_stderr':matched.observable_stderr['Z'],
        'status':'representative_large_strip'})
    for phase_check_value in LARGE_STRIP_PHASE_CHECK_VALUES:
        phase_check_value=float(phase_check_value)
        if np.isclose(phase_check_value,CHECKERBOARD_REPRESENTATIVE_PHASE):
            continue
        check_instance=checkerboard_instance(int(repeats),phase_check_value)
        check_build=check_instance.model.build(basis=basis,builder='bitmask',backend='scipy',sort_basis=False)
        check_h_sector=project_sparse_operator_to_sector(check_build.hamiltonian,sector)
        check_scan=canonical_typicality_scan(check_h_sector,projected_q,beta_max=FINITE_BETA_BETA_MAX,
            beta_points=FINITE_BETA_BETA_POINTS,n_samples=FINITE_BETA_TYPICALITY_SAMPLES,
            random_seed=FINITE_BETA_RANDOM_SEED+Lx)
        check_match=energy_matched_canonical_estimate(check_scan,target_energy=tower_energy)
        check_gradient=np.gradient(check_scan.energy,check_scan.beta)
        check_index=int(np.argmin(np.abs(check_scan.beta-check_match.beta)))
        check_beta_stderr=check_match.energy_stderr/max(1e-12,abs(check_gradient[check_index]))
        large_phase_rows.append({'Lx':Lx,'phase':phase_check_value,'method':'canonical_typicality_common_sector',
            'beta_raw':check_match.beta,'tau_A_raw':check_match.observables['A'],'tau_Z_raw':check_match.observables['Z'],
            'beta_stderr':check_beta_stderr,'tau_A_stderr':check_match.observable_stderr['A'],
            'tau_Z_stderr':check_match.observable_stderr['Z'],'status':'large_strip_phase_check'})
    print({'large_strip_stage':'shift_invert','eigenpairs':LARGE_STRIP_EIGENPAIRS})
    partial=shift_invert_partial_spectrum(h_sector,target_energy=tower_energy,eigenpairs=LARGE_STRIP_EIGENPAIRS,
        sigma_offset=LARGE_STRIP_SIGMA_OFFSET,tolerance=LARGE_STRIP_EIG_TOL,maxiter=LARGE_STRIP_MAXITER)
    q_all=translated_joint_dark(configs,model,sector); exceptional,jrows=joint_dark_kernel(partial.energies,partial.eigenvectors,q_all,tower)
    for row in jrows:
        row.update({'repeats':repeats,'Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,'spectrum_method':'shift_invert_partial'})
    large_dark_rows.extend(jrows)
    resolved=projector_resolved_energy_basis(partial.energies,partial.eigenvectors,exceptional,
        energy_tolerance=ENERGY_BLOCK_TOL,vector_tolerance=1e-9)
    keep=~resolved['is_exceptional'].astype(bool); clean_E=resolved['energies'][keep]
    clean_Q={name:np.real(np.einsum('ij,ij->j',resolved['basis'][:,keep].conj(),op@resolved['basis'][:,keep])) for name,op in projected_q.items()}
    for pref in MICROCANONICAL_PREFACTORS:
        plan=thermodynamic_energy_window_plan(volume=volume,energy_density=tower_energy/volume,width_prefactor=pref,local_energy_scale=1.)
        coverage=partial.covers_window(plan.half_width,margin=10*ENERGY_BLOCK_TOL)
        if coverage:
            window=select_microcanonical_window_by_width(clean_E,target_energy=tower_energy,half_width=plan.half_width,degeneracy_tolerance=ENERGY_BLOCK_TOL)
            idx=np.asarray(window.indices,int); mc={name:float(np.mean(vals[idx])) for name,vals in clean_Q.items()}
            clean_count=window.n_states
        else:
            mc={'A':np.nan,'Z':np.nan}; clean_count=0
        large_strip_rows.append({'repeats':repeats,'Lx':Lx,'Ly':4,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
            'thermal_protocol':'finite-beta','matched_beta':matched.beta,'sector_dimension':sector.sector_dimension,
            'cage_energy':tower_energy,'cage_energy_density':tower_energy/volume,'cage_residual':tower_residual,
            'window_prefactor':pref,'window_half_width':plan.half_width,'window_energy_density_half_width':plan.energy_density_half_width,
            'window_state_count':clean_count,'window_coverage_complete':coverage,'spectrum_method':'shift_invert_partial',
            'partial_eigenpair_count':partial.energies.size,'partial_min_energy':partial.min_energy,'partial_max_energy':partial.max_energy,
            'partial_maximum_residual':partial.maximum_residual,'reference_method':'canonical_typicality_common_sector',
            'reference_cleaning':'clean_canonical_pending_dark_manifold_classification','joint_dark_rank':exceptional.shape[1],
            'removed_fraction':float(exceptional.shape[1]/max(1,partial.energies.size)),'raw_window_state_count':np.nan,
            'clean_window_state_count':clean_count,'tau_A_mc':mc['A'],'tau_Z_mc':mc['Z'],'tau_A_mc_raw':np.nan,'tau_Z_mc_raw':np.nan,
            'tau_A_reference':np.nan,'tau_Z_reference':np.nan,
            'tau_A_reference_raw':matched.observables['A'],'tau_Z_reference_raw':matched.observables['Z'],
            'tau_A_reference_clean':np.nan,'tau_Z_reference_clean':np.nan,
            'tau_A_reference_physical':matched.observables['A'],'tau_Z_reference_physical':matched.observables['Z'],
            'matched_beta_raw':matched.beta,'delta_A':np.nan,'delta_Z':np.nan,'Delta':np.nan,
            'delta_A_clean_clean':np.nan,'delta_Z_clean_clean':np.nan,
            'delta_A_physical_target':abs(mc['A']-matched.observables['A']) if coverage else np.nan,
            'delta_Z_physical_target':abs(mc['Z']-matched.observables['Z']) if coverage else np.nan,
            'Delta_physical_target':max(abs(mc['A']-matched.observables['A']),abs(mc['Z']-matched.observables['Z'])) if coverage else np.nan,
            'cage_QA':q_res['A'],'cage_QZ':q_res['Z']})
    primary_plan=thermodynamic_energy_window_plan(volume=volume,energy_density=tower_energy/volume,
        width_prefactor=PRIMARY_WINDOW_PREFACTOR,local_energy_scale=1.)
    if partial.covers_window(primary_plan.half_width,margin=10*ENERGY_BLOCK_TOL):
        rep_scatter=pd.DataFrame({'repeats':repeats,'Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,'energy':clean_E,
            'energy_density':clean_E/volume,'Q_A':clean_Q['A'],'Q_Z':clean_Q['Z'],'is_tower_state':False})
        large_scatter_frames.append(rep_scatter)
        if RUN_CHECKERBOARD_CONCENTRATION:
            ops,names,meta=stripe_algebra(configs,model,sector)
            raw_window=select_microcanonical_window_by_width(partial.energies,target_energy=tower_energy,
                half_width=primary_plan.half_width,degeneracy_tolerance=ENERGY_BLOCK_TOL)
            covariance=projector_deleted_block_covariance(partial.energies,partial.eigenvectors,exceptional,ops,
                np.asarray(raw_window.indices,int),energy_tolerance=ENERGY_BLOCK_TOL,vector_tolerance=1e-9)
            large_concentration_rows.append({'repeats':repeats,'Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
                'operator_space_dimension':len(ops),**meta,'largest_covariance_eigenvalue':covariance['largest_eigenvalue'],
                'w':covariance['largest_width'],'median_nonidentity_width':covariance['median_nonidentity_width'],
                'energy_block_tolerance':ENERGY_BLOCK_TOL,'spectrum_method':'shift_invert_partial','window_coverage_complete':True,
                'partial_eigenpair_count':partial.energies.size,'worst_coefficients':repr(dict(zip(names,np.asarray(covariance['worst_coefficients']).tolist(),strict=True)))})

if large_strip_rows:
    thermal=pd.concat([thermal,pd.DataFrame(large_strip_rows)],ignore_index=True)
    thermal.to_csv(DATA_DIR/'qdm_checkerboard_thermal_overlap.csv',index=False)
    thermal.to_csv(DATA_DIR/'qdm_checkerboard_window_systematics.csv',index=False)
if large_dark_rows:
    dark=pd.concat([dark,pd.DataFrame(large_dark_rows)],ignore_index=True); dark.to_csv(DATA_DIR/'qdm_checkerboard_joint_dark_kernel.csv',index=False)
if large_concentration_rows:
    concentration=pd.concat([concentration,pd.DataFrame(large_concentration_rows)],ignore_index=True)
    concentration.to_csv(DATA_DIR/'qdm_checkerboard_concentration_grid.csv',index=False)
    concentration.to_csv(DATA_DIR/'qdm_checkerboard_worst_eigenoperator.csv',index=False)
if large_scatter_frames:
    scatter=pd.concat([scatter,*large_scatter_frames],ignore_index=True)
    largest=int(scatter[np.isclose(scatter.phase,CHECKERBOARD_REPRESENTATIVE_PHASE)].Lx.max())
    scatter[(scatter.Lx==largest)&np.isclose(scatter.phase,CHECKERBOARD_REPRESENTATIVE_PHASE)].to_csv(
        DATA_DIR/'qdm_checkerboard_eth_scatter.csv',index=False)
if large_phase_rows:
    phase_path=DATA_DIR/'qdm_checkerboard_finite_beta_transfer_phase_check.csv'
    existing=pd.read_csv(phase_path) if phase_path.exists() else pd.DataFrame()
    pd.concat([existing,pd.DataFrame(large_phase_rows)],ignore_index=True,sort=False).to_csv(phase_path,index=False)
finite_beta_target.to_csv(DATA_DIR/'qdm_checkerboard_finite_beta_transfer_target.csv',index=False)
finite_beta_target[['Lx','target_energy_density','beta_star','method','status']].to_csv(
    DATA_DIR/'qdm_checkerboard_finite_beta_energy_match.csv',index=False)


## P0.5--P0.6: controlled fits and representative phase

Thermodynamic fits are emitted only after at least three controlled lengths are available. With fewer lengths, the tables explicitly report `insufficient_lengths`. The representative phase remains provisional until the compact dark manifold and third-size thermal/concentration diagnostics pass.


In [ ]:
def fit_sequence(frame,value_column,*,minimum_lengths=3):
    frame=frame.dropna(subset=['Lx',value_column]).sort_values('Lx')
    L=np.asarray(frame.Lx,float); y=np.asarray(frame[value_column],float); rows=[]
    if np.unique(L).size<minimum_lengths:
        return pd.DataFrame([{'fit_form':'none','included_Lx':','.join(map(str,L.astype(int))),
            'limit':np.nan,'slope':np.nan,'rmse':np.nan,'status':'insufficient_lengths'}])
    for name,x,free in [('Delta_inf+c/Lx',1/L,True),('Delta_inf+c/Lx^2',1/L**2,True),('c/Lx',1/L,False),('c/Lx^2',1/L**2,False)]:
        if free:
            slope,intercept=np.polyfit(x,y,1); pred=intercept+slope*x
        else:
            intercept=0.; slope=float(np.dot(x,y)/np.dot(x,x)); pred=slope*x
        rows.append({'fit_form':name,'included_Lx':','.join(map(str,L.astype(int))),'limit':float(intercept),
            'slope':float(slope),'rmse':float(np.sqrt(np.mean((y-pred)**2))),'status':'diagnostic_fit'})
    return pd.DataFrame(rows)

# Large-L canonical target fits use the physical raw canonical trace.
if not finite_beta_target.empty:
    finite_size=finite_beta_target[finite_beta_target.record_type=='finite_size'].drop_duplicates('Lx').sort_values('Lx')
    target_fit_rows=[]
    for quantity in ('beta_star','tau_A_target','tau_Z_target'):
        fits=fit_sequence(finite_size,quantity)
        for _,row in fits.iterrows(): target_fit_rows.append({'record_type':'fit','quantity':quantity,**row.to_dict()})
    target_fits=pd.DataFrame(target_fit_rows)
    if not target_fits.empty: finite_beta_target=pd.concat([finite_beta_target,target_fits],ignore_index=True,sort=False)
    finite_beta_target.to_csv(DATA_DIR/'qdm_checkerboard_finite_beta_transfer_target.csv',index=False)

primary=thermal[np.isclose(thermal.window_prefactor,PRIMARY_WINDOW_PREFACTOR)].copy() if not thermal.empty else pd.DataFrame()
fit_rows=[]; concentration_fit_rows=[]
if not primary.empty:
    positive=primary[primary.phase.isin(CHECKERBOARD_POSITIVE_PHASE_VALUES)&primary.window_coverage_complete.astype(bool)]
    rep_primary=positive[np.isclose(positive.phase,CHECKERBOARD_REPRESENTATIVE_PHASE)]
    for _,row in fit_sequence(rep_primary,'Delta').iterrows():
        fit_rows.append({'scope':'representative_clean_clean_diagnostic','phase':CHECKERBOARD_REPRESENTATIVE_PHASE,**row.to_dict()})
    for _,row in fit_sequence(rep_primary,'Delta_physical_target').iterrows():
        fit_rows.append({'scope':'representative_physical_canonical','phase':CHECKERBOARD_REPRESENTATIVE_PHASE,**row.to_dict()})
    if not positive.empty:
        clean_envelope=positive.groupby('Lx',as_index=False).Delta.max()
        for _,row in fit_sequence(clean_envelope,'Delta').iterrows():
            fit_rows.append({'scope':'positive_grid_clean_clean_diagnostic','phase':np.nan,**row.to_dict()})
        physical_envelope=positive.groupby('Lx',as_index=False).Delta_physical_target.max()
        for _,row in fit_sequence(physical_envelope,'Delta_physical_target').iterrows():
            fit_rows.append({'scope':'positive_grid_physical_canonical','phase':np.nan,**row.to_dict()})
    if not concentration.empty:
        conc=concentration[concentration.phase.isin(CHECKERBOARD_POSITIVE_PHASE_VALUES)&concentration.window_coverage_complete.astype(bool)]
        rep_conc=conc[np.isclose(conc.phase,CHECKERBOARD_REPRESENTATIVE_PHASE)]
        for _,row in fit_sequence(rep_conc,'w').iterrows(): concentration_fit_rows.append({'scope':'representative','phase':CHECKERBOARD_REPRESENTATIVE_PHASE,**row.to_dict()})
        if not conc.empty:
            envelope=conc.groupby('Lx',as_index=False).w.max()
            for _,row in fit_sequence(envelope,'w').iterrows(): concentration_fit_rows.append({'scope':'positive_grid_envelope','phase':np.nan,**row.to_dict()})
pd.DataFrame(fit_rows).to_csv(DATA_DIR/'qdm_checkerboard_matching_distance_fit.csv',index=False)
pd.DataFrame(fit_rows).to_csv(DATA_DIR/'qdm_checkerboard_fixed_width_shared_fit.csv',index=False)
pd.DataFrame(concentration_fit_rows).to_csv(DATA_DIR/'qdm_checkerboard_uniform_concentration_fit.csv',index=False)

if not primary.empty:
    candidates=primary[primary.phase.isin(CHECKERBOARD_POSITIVE_PHASE_VALUES)]
    available=sorted(candidates.phase.unique())
    phi_star=CHECKERBOARD_REPRESENTATIVE_PHASE if CHECKERBOARD_REPRESENTATIVE_PHASE in available else available[len(available)//2]
    classification_ok=(not dark_vs_type1.empty and dark_vs_type1.all_joint_dark_explained.fillna(False).all())
    covered=candidates[candidates.window_coverage_complete.astype(bool)]
    third_length=bool(covered.dropna(subset=['Delta_physical_target']).Lx.nunique()>=3)
    clean_third_length=bool(covered.dropna(subset=['Delta']).Lx.nunique()>=3)
    concentration_third=bool(not concentration.empty and concentration[concentration.window_coverage_complete.astype(bool)].Lx.nunique()>=3)
    selection_status='finalized_physical_target' if classification_ok and third_length and concentration_third else 'provisional_pending_P0'
    conc_at_phi=concentration[np.isclose(concentration.phase,phi_star)] if not concentration.empty else pd.DataFrame()
    pd.DataFrame([{'phi_star':phi_star,'selection':selection_status,'thermal_protocol':'finite-beta',
        'dark_manifold_classified':classification_ok,'third_energy_resolved_length_available':third_length,'third_clean_clean_length_available':clean_third_length,
        'third_concentration_length_available':concentration_third,
        'maximum_sampled_Delta':float(candidates[np.isclose(candidates.phase,phi_star)].Delta.max()),
        'maximum_sampled_concentration_width':float(conc_at_phi.w.max()) if not conc_at_phi.empty else np.nan}]).to_csv(
        DATA_DIR/'qdm_checkerboard_representative_phase.csv',index=False)
    print({'representative_phase':phi_star,'selection':selection_status,'third_length':third_length,'dark_classified':classification_ok})
else:
    print('Thermal scan unavailable; Fig. 7 remains provisioned.')


## Output and claim manifest

The manifest records only the strength supported by completed gates. The failed $\beta=0$ route is retained as a control and is not promoted back into the primary finite-temperature claim.


In [ ]:
common_sector=pd.read_csv(DATA_DIR/'qdm_checkerboard_common_symmetry_sector.csv') if (DATA_DIR/'qdm_checkerboard_common_symmetry_sector.csv').exists() else pd.DataFrame()
sector12=bool(not common_sector.empty and ((common_sector.Lx==12)&(common_sector.status=='verified_sparse_large_strip')).any())
covered_thermal=thermal[thermal.window_coverage_complete.astype(bool)] if not thermal.empty else pd.DataFrame()
third_length=bool(not covered_thermal.empty and covered_thermal.dropna(subset=['Delta_physical_target']).Lx.nunique()>=3)
third_clean_length=bool(not covered_thermal.empty and covered_thermal.dropna(subset=['Delta']).Lx.nunique()>=3)
dark_classified=bool(not dark_vs_type1.empty and dark_vs_type1.all_joint_dark_explained.fillna(False).all())
gate_rows=[
 {'gate':'1_energy_density','status':'failed_beta0_switch_finite_beta','detail':f'beta0 mismatch limit={preferred:.6g}'},
 {'gate':'2_local_transport','status':'passed' if GATE2_LOCAL_PASSED else 'failed','detail':'4x4,8x4,12x4 local certificates'},
 {'gate':'P0.1_finite_beta_target','status':'three_length_provisional' if finite_beta_target.Lx.dropna().nunique()>=3 else 'fewer_than_three_lengths',
  'detail':'exact ED at small strips; canonical typicality on 12x4 when enabled'},
 {'gate':'P0.2_dark_manifold','status':'classified' if dark_classified else 'classification_pending_or_incomplete','detail':'direct Type-I span versus translated A,Z joint-dark kernel'},
 {'gate':'P0.3_common_sector_12x4','status':'passed' if sector12 else 'pending_large_strip_run','detail':'sparse T_x^2,T_y^2 k=0 projection'},
 {'gate':'P0.4_third_energy_resolved','status':'passed_physical_target_partial_spectrum' if third_length else 'pending_large_strip_run_or_window_coverage','detail':'shift-invert must cover the declared window; clean--clean third length remains separately audited'},
 {'gate':'P0.5_clean_clean_third_length','status':'passed' if third_clean_length else 'pending_clean_canonical_large_strip','detail':'legacy clean--clean estimator is not mixed with raw typicality'},]
pd.DataFrame(gate_rows).to_csv(DATA_DIR/'qdm_checkerboard_scientific_gates.csv',index=False)
claim_status='sampled_fixed_width_evidence_ready_for_draft_review' if third_length and dark_classified else 'candidate_sampled_compatibility_finite_beta_pending'
pd.DataFrame([{'claim_id':'checkerboard_family','status':claim_status,'thermal_protocol':'finite-beta',
 'third_energy_resolved_length_available':third_length,'third_clean_clean_length_available':third_clean_length,'dark_manifold_classified':dark_classified,
 'note':'No fixed-width thermodynamic claim unless the finite-beta target, third-length matching, concentration, and dark-manifold checks are controlled.'}]).to_csv(DATA_DIR/'claim_manifest.csv',index=False)
write_figure_manifest(DATA_DIR/'figure_manifest.json')
for path in sorted(DATA_DIR.rglob('*')):
    if path.is_file(): print(path.relative_to(DATA_DIR))
